# NBA en diferido · sin spoilers
Solo se muestra el enfrentamiento y **ver completo**, **resumen** o **puedes saltártelo**, en orden alfabético.

Ejecuta las celdas en orden. Si faltan dependencias: `%pip install requests tzdata`.

La consulta evalúa partidos finalizados cuyo inicio esté entre ayer 06:30 y hoy 06:30 Madrid. A las 05:00 la cobertura puede ser incompleta. No hay envío de correo desde el cuaderno.

Criterio v2: competitividad gradual `exp(-margen/14)` con peso 80%; afinidad con peso 20%, normalizada como `0,5 + (afinidad local + afinidad visitante)/4`; Celtics añade 0,04. El total se limita a 1. Umbrales internos: completo >=0,74; resumen >=0,50; resto puedes saltártelo. No se muestran notas ni señales del desenlace. Sin premio por anotación ni clasificación inventada. El margen sigue siendo una aproximación: no representa la evolución completa del partido. Afinidades originales conservadas.

Fuente: endpoint público ESPN; puede cambiar. Sin credenciales. Ningún resultado queda guardado en el cuaderno distribuido.

Preferencia personal: los partidos finalizados de Celtics siempre se recomiendan como Resumen, independientemente de la nota interna. Sus tarjetas son verdes.


In [ ]:
"""NBA en diferido: la salida pública no contiene marcadores ni notas."""
import json
import math
import time
from datetime import datetime, timedelta, time as hora, timezone
from pathlib import Path

import requests
from zoneinfo import ZoneInfo
MADRID_TZ = ZoneInfo("Europe/Madrid")

URL = 'https://site.api.espn.com/apis/site/v2/sports/basketball/nba/scoreboard'
AFINIDADES = {'Celtics': 0.85, 'Warriors': 0.75, 'Nuggets': 0.7, 'Timberwolves': 0.7, 'Spurs': 0.6, 'Rockets': 0.6, 'Pacers': 0.55, 'Raptors': 0.5, 'Hornets': 0.45, 'Pelicans': 0.35, 'Clippers': 0.3, 'Kings': 0.3, 'Bucks': 0.2, 'Hawks': 0.15, 'Nets': 0.15, 'Bulls': 0.1, 'Magic': 0.05, 'Wizards': 0.0, 'Pistons': 0.0, 'Jazz': 0.0, 'Trail Blazers': 0.0, 'Thunder': -0.45, 'Knicks': -0.4, 'Heat': -0.4, '76ers': -0.35}


class ErrorNBA(Exception):
    pass


def descargar(dia):
    for intento in range(3):
        try:
            r = requests.get(URL, params={'dates': dia.strftime('%Y%m%d'), 'limit': 100}, timeout=(5, 20))
            if r.status_code == 429 or r.status_code >= 500:
                if intento < 2:
                    time.sleep(2 ** intento)
                    continue
            r.raise_for_status()
            datos = r.json()
            if not isinstance(datos, dict) or not isinstance(datos.get('events'), list):
                raise ErrorNBA('La fuente NBA ha cambiado de formato.')
            return datos['events']
        except (requests.Timeout, requests.ConnectionError):
            if intento < 2:
                time.sleep(2 ** intento)
                continue
            raise ErrorNBA('No se pudo consultar la NBA tras tres intentos.') from None
        except (requests.RequestException, ValueError):
            raise ErrorNBA('No se pudo leer la fuente NBA.') from None


def puntuacion(local, visitante, puntos_local, puntos_visitante):
    diferencia = abs(puntos_local - puntos_visitante)
    competitividad = math.exp(-diferencia / 14)
    gustos = max(0, min(1, .5 + (AFINIDADES.get(local, 0) + AFINIDADES.get(visitante, 0)) / 4))
    favorito = .04 if 'Celtics' in (local, visitante) else 0
    # Sin puntos totales ni clasificación ficticia; escala interna, nunca publicada.
    return min(1, .80 * competitividad + .20 * gustos + favorito)


def categoria_nba(nota):
    return 'Ver completo' if nota >= .74 else 'Resumen' if nota >= .50 else 'Puedes saltártelo'



def evaluar_partidos(partidos, ahora):
    if ahora.tzinfo is None:
        raise ValueError('La fecha NBA debe incluir zona horaria.')
    ahora = ahora.astimezone(MADRID_TZ)
    cierre = datetime.combine(ahora.date(), hora(6, 30), MADRID_TZ)
    inicio = datetime.combine(ahora.date() - timedelta(days=1), hora(6, 30), MADRID_TZ)
    limite = min(ahora, cierre)
    vistos = set()
    salida, pendientes, invalidos = [], 0, 0
    for e in partidos:
        try:
            fecha = datetime.fromisoformat(e['date'].replace('Z', '+00:00'))
            if fecha.tzinfo is None:
                raise ValueError()
            if not inicio <= fecha < cierre or fecha > limite:
                continue
            identificador = str(e['id'])
            if identificador in vistos:
                continue
            vistos.add(identificador)
            if not e['status']['type']['completed']:
                pendientes += 1
                continue
            equipos = e['competitions'][0]['competitors']
            local = next(t for t in equipos if t['homeAway'] == 'home')
            visitante = next(t for t in equipos if t['homeAway'] == 'away')
            puntos = [int(t['score']) for t in (local, visitante)]
            if any(p < 0 for p in puntos) or puntos[0] == puntos[1]:
                raise ValueError()
            nombres = [t['team']['displayName'] for t in (local, visitante)]
            afinidades = [t['team']['name'] for t in (local, visitante)]
            nota = puntuacion(*afinidades, *puntos)
            recomendacion = 'Resumen' if 'Celtics' in afinidades else categoria_nba(nota)
            # Nunca devolver el objeto de la fuente: contiene spoilers incluso en sus enlaces.
            salida.append({'local': nombres[0], 'visitante': nombres[1],
                           'fecha': fecha.astimezone(MADRID_TZ).strftime('%d/%m %H:%M'),
                           'recomendacion': recomendacion})
        except (KeyError, IndexError, TypeError, ValueError, StopIteration, AttributeError):
            invalidos += 1
    # Orden estable por nombres, nunca por margen/nota oculta.
    salida.sort(key=lambda p: (p['local'], p['visitante']))
    avisos = []
    if pendientes:
        avisos.append(f'{pendientes} encuentros iniciados aún sin resultado final confirmado; no se evalúan.')
    if invalidos:
        avisos.append(f'{invalidos} registros no se pudieron validar; cobertura incompleta.')
    if ahora < cierre:
        avisos.append('Consulta antes de las 06:30: la jornada puede seguir en curso y el correo no se actualiza automáticamente.')
    return {'partidos': salida, 'avisos': avisos, 'fuente': 'ESPN',
            'ventana': f'{inicio:%d/%m %H:%M}–{cierre:%d/%m %H:%M} · Madrid',
            'consultado': ahora.strftime('%d/%m %H:%M')}


def generar_nba(ahora=None):
    ahora = ahora or datetime.now(MADRID_TZ)
    if ahora.tzinfo is None:
        raise ValueError('La fecha NBA debe incluir zona horaria.')
    local = ahora.astimezone(MADRID_TZ)
    inicio = datetime.combine(local.date() - timedelta(days=1), hora(6, 30), MADRID_TZ)
    cierre = min(local, datetime.combine(local.date(), hora(6, 30), MADRID_TZ))
    dia = inicio.astimezone(timezone.utc).date()
    ultimo = cierre.astimezone(timezone.utc).date()
    partidos = []
    while dia <= ultimo:
        partidos.extend(descargar(dia))
        dia += timedelta(days=1)
    return evaluar_partidos(partidos, ahora)

"""Identidad visual: equipos por delante de competición, sin inferir resultados."""
import unicodedata


def normalizar(texto):
    return ' '.join(''.join(c for c in unicodedata.normalize('NFD', texto.casefold())
                           if not unicodedata.combining(c)).split())


def tema_evento(e):
    equipos = {normalizar(e.get(k, '')) for k in ('local', 'visitante')}
    deporte = normalizar(e.get('deporte', ''))
    comp = normalizar(e.get('competicion', ''))
    if deporte == 'baloncesto' and equipos & {'celtics', 'boston celtics'}:
        return '#e2f3e8', '#007a33', 'Boston Celtics'
    if deporte == 'futbol':
        if equipos & {'atletico de madrid', 'atletico madrid', 'at. madrid', 'at madrid', 'espana', 'seleccion espanola'}:
            return '#ffe5e5', '#b51c2b', 'Atleti / España'
        if 'aston villa' in equipos:
            return '#f4e3ea', '#760d35', 'Aston Villa'
        if 'conference' in comp:
            return '#e7f4df', '#39751c', 'Conference League'
        if 'europa league' in comp or 'liga europa' in comp:
            return '#ffecd9', '#aa4b08', 'Europa League'
        if 'champions' in comp or 'liga de campeones' in comp:
            return '#e4ebff', '#183d96', 'Champions League'
        return '#e2f1f0', '#286c69', 'Fútbol'
    if deporte == 'tenis':
        return '#f1f7b5', '#566400', 'Tenis'
    if deporte == 'baloncesto':
        if 'nba' in comp:
            return '#ffdfc1', '#b45309', 'NBA'
        if 'euroliga' in comp or 'euroleague' in comp:
            return '#ede4fa', '#683d99', 'Euroliga'
        if any(s in comp for s in ('acb', 'endesa', 'copa del rey')):
            return '#dff0ff', '#176a91', 'ACB'
        return '#fff0df', '#955717', 'Baloncesto'
    return '#edf2f7', '#52647a', e.get('deporte', 'Deporte')


In [ ]:
from IPython.display import HTML, display
from html import escape
nba = generar_nba()
filas = ''
for p in nba['partidos']:
    fondo, acento, _ = tema_evento({**p, 'deporte':'Baloncesto', 'competicion':'NBA'})
    filas += f'<div style="padding:16px;margin:12px 0;background:{fondo};border-left:5px solid {acento};border-radius:10px"><strong>' + escape(p['recomendacion']) + '</strong><p>' + escape(p['local'] + ' — ' + p['visitante']) + '</p></div>'

if not filas:
    filas = '<p>No hay partidos finalizados y validados en esta consulta.</p>'
cobertura = '<br>'.join(escape(x) for x in [nba['ventana'], 'Consultado: ' + nba['consultado'] + ' · Fuente: ESPN', *nba['avisos']])
display(HTML('<div style="background:#fff2e5;color:#142840;border:2px solid #c45a12;border-radius:16px;overflow:hidden"><div style="background:#142840;color:white;padding:20px;border-bottom:5px solid #ed7b24"><h2>🏀 NBA en diferido</h2><p>Sin spoilers · Orden alfabético</p></div><div style="padding:16px">' + filas + '<p style="font-size:12px">' + cobertura + '</p></div></div>'))
